# JAXVacua: overview

**What's in this notebook?** This notebook summarises the basic capabilities of JAXVacua.

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

### General imports

In [1]:
import sys, os, warnings
import numpy as np
from tqdm.auto import tqdm
from functools import partial
from typing import Any, Callable, Sequence
from IPython.display import clear_output

warnings.filterwarnings('ignore')

### JAX imports

In [2]:
from jax import jit, vmap
import jax 
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

### Plotting tools

In [3]:
import seaborn as sn
import matplotlib.pyplot as plt
import matplotlib as mpl
cmap=sn.color_palette("viridis", as_cmap=True)

### Custom library for EFT

In [4]:
sys.path.append("./../../../")
import jaxvacua

## Introduction to JAXVacua  

[JAXVacua](https://arxiv.org/pdf/2306.06160) is a software package designed for the efficient exploration of string theory vacua, leveraging the computational power of JAX. Developed as a tool for high-dimensional optimisation and machine learning applications in theoretical physics, JAXVacua provides a scalable and flexible framework for analysing complex energy landscapes in string theory and related fields.  

Key features of JAXVacua include:  
- **Automatic Differentiation**: Utilising JAX’s autograd capabilities to compute gradients and Hessians efficiently, aiding in the search for vacua.  
- **Just-In-Time (JIT) Compilation**: Enhancing performance through XLA-based compilation for CPU, GPU, and TPU acceleration.  
- **Vectorisation (`vmap`)**: Enabling simultaneous evaluation of multiple candidate vacua, significantly improving computational efficiency.  
- **Parallelism (`pmap`)**: Allowing large-scale parallel computations across multiple devices, making it suitable for extensive searches in high-dimensional parameter spaces.  

JAXVacua is particularly well-suited for studies in flux compactifications, moduli stabilisation, and other problems requiring rapid and scalable numerical analysis. By integrating modern machine learning techniques with traditional methods in string theory, it provides a powerful computational framework for advancing research in fundamental physics.

To reiterate, moduli stabilisation requires finding vacua where the scalar potential, derived from fluxes, branes, and other ingredients in string theory, has local minima. As we argue below, **automatic differentiation** can play a crucial role in this process by efficiently computing gradients and Hessians of the scalar potential. The first derivatives (gradients) help identify critical points where the potential is stationary, while second derivatives (Hessians) determine their stability by checking for positive eigenvalues (indicating local minima). Given the high-dimensional and complex nature of the moduli space, manual differentiation is infeasible, and numerical methods like finite differences introduce errors. AD provides exact derivatives up to machine precision without symbolic computation overhead, making it highly efficient.

Frameworks like JAX further enhance this process by enabling rapid gradient computation (`jax.grad`), second-order differentiation (`jax.hessian`), and Just-In-Time (JIT) compilation for acceleration. These features allow for efficient optimization and exploration of the moduli space, making AD a powerful tool for string compactifications and moduli stabilisation. We will introduce this process further below. 

## Sub-modules and basic capabilities

Let us describe one example in some more detail.
First, we provide different options to load examples:
* from file: we provide files for selected models from the Kreuzer-Skarke (`KS`) list for variying $h^{1,2}$ as well as for all Complete Intersection CY threefolds (`CICYs`). These models can be accessed by specifying $h^{1,2}$ and a corresponding `model_ID`.
* from input: if either periods or other relevant information specifying the periods are known, they can be provided directly. E.g., at LCS, the intersection numbers of the mirror CY as well as some other topological information suffices to completely specify the periods.
* from CYToools: for `KS` models at LCS, we can simply provide a `cytools.CalabiYau` object to compute the relevant topological data of the mirror CY.

The package consists of mainly four submodules which we describe in details below:
* ```jaxvacua.periods```
* ```jaxvacua.css```
* ```jaxvacua.flux_sector```
* ```jaxvacua.sampling```

The are two additional submodules which should however not be called directly:
* ```jaxvacua.periods_LCS```: contains functions to compute the period sector at LCS.
* ```jaxvacua.css_LCS```: contains functions to compute the complex structure at LCS.

These two classes are useful to compute periods for a large fraction of models, see in particular the notebooks [3_cytools_interface](./3_cytools_interface.ipynb) and [4_cicy](./4_cicy.ipynb).
Beyond periods at LCS, our implementation allows users to easily include their own custom period class for different regions in (complex structure) moduli space.


### `periods` sub-module

This sub-module contains `periods` class implementing standard formulas in terms of the periods.

The holomorphic $3$-form $\Omega$ on $X$ can be parametrised by the real, symplectic basis of $3$-forms $(\alpha_I, \beta^J)\in H_{-}^{3}(X_{3},\mathbb{Z})$, $I,J\in \{0, \ldots, h^{1,2}_-(X)\}$, together with a dual basis of $3$-cycles $(A_{I},B^{J})\in H^{-}_{3}(X_{3},\mathbb{Z})$ so that

$$
\Omega_{3}=  X^I \, \alpha_I - \,  \mathcal{F}_{J} \,  \beta^J\; ,\quad  X^{I}= \int_{A_{I}}\, \Omega_{3}\; ,\quad  F_{J}= \int_{B^{J}}\, \Omega_{3}  \, .
$$ 

Here, $(X^{I},\mathcal{F}_{J})$ are the so-called \emph{periods} of $\Omega_{3}$ which are usually arranged in the \emph{period vector}\index{Period vector} 

$$
\Pi=\left (\begin{array}{c}\mathcal{F}_{J}\\[0.3em]
X^{I}\end{array} \right )\, .
$$


Due to the *special* Kähler structure of $\mathcal{M}_{\text{cs}}(X)$,
half of the periods can be obtained from a *prepotential* $F(X^I)$, that is,

$$
\Pi=\left (\begin{array}{c}
 \partial_{X^I}F\\ 
    X^{I}
\end{array} \right )\, .
$$




Another readily available function that can be computed from knowing the periods is the **gauge kinetic matrix**.
$$
\mathcal{N}_{IJ} = (\mathcal{F}_{I} ,\, D_{\bar{\imath}}\overline{\mathcal{F}}_I )\, \cdot\, (X^{J},\, D_{\bar{\jmath}}\overline{X}^J)^{-1}= P_{IK} \, (Q^{-1})^{K}\,_{J}\quad , \; P_{IJ} = (\mathcal{F}_{I} ,\, D_{\bar{\imath}}\overline{\mathcal{F}}_I )_{J}\quad , \; Q^{I}\,_{J} = (X^{I},\, D_{\bar{\jmath}}\overline{X}^I)_{J}\, ,                               
$$
see e.g. Sect. 2 above Eq. (2.12) in [2110.05511](https://inspirehep.net/files/14d463478651c33420dee54ab42be7c6) or Eq. (3.5) in [2310.06040](https://arxiv.org/abs/2310.06040).
If we are working with a prepotential $F$, the gauge kinetic matrix can also be computed using
$$
\mathcal{N}_{I J}=\overline{F}_{IJ}+2\text{i}\, \dfrac{\text{Im}(F_{I L})X^{L} \, \text{Im}(F_{J K})X^{K}}{X^{M}\text{Im}(F_{MN})X^{N}}\; ,\quad  F_{IJ}= \partial_{X^{I}} \partial_{X^{J}}F\, .
$$
Both options are implemented in our code, see `periods.gauge_kinetic_matrix_periods` and `periods.gauge_kinetic_matrix_prepotential`.

By writing $\mathcal{N}=\mathcal{R}+\text{i} \mathcal{I}$, the Hodge-star matrix $\mathcal{M}$ is given by
$$
\mathcal{M}=\left (\begin{array}{cc}
 -\mathcal{I}^{-1} \hspace{0.3cm}&\mathcal{I}^{-1}\mathcal{R} \\ 
 \mathcal{R}\mathcal{I}^{-1}  & -\mathcal{I}-\mathcal{R}\mathcal{I}^{-1}\mathcal{R}
\end{array} \right )\, .
$$
This function can be called via ```periods.ISD_matrix```.
Note that our convention differs slightly from that of Eq. (3.7) in [2310.06040](https://arxiv.org/abs/2310.06040) due to the ordering of periods in the period vector. Specifically, we have
$$
                \mathcal{M}_{\mathrm{here}} = \mathcal{M}_{\mathrm{there}}^{-1}\, .
$$


#### Large Complex Structure limits

At Large Complex Structure (LCS), the prepotential can be expressed as

$$
    F_{\mathrm{LCS}}(X) = F_{\mathrm{poly}}(X) + F_{\mathrm{inst}}(X)\, .
$$

Here, the polynomial part $F_{\mathrm{poly}}$ of the LCS prepotential $F_{\mathrm{LCS}}$
can be expressed in terms of the periods $X^I=(X^0,X^i)$ as

$$
    F_{\mathrm{poly}}(X) = -\frac{1}{6X^0}\widetilde{\kappa}_{ijk}X^iX^jX^k+\frac{1}{2}a_{ij}X^iX^j
        +b_{i}X^i\, X^0 + \dfrac{\text{i}}{2}\tilde{\xi}\, (X^0)^2\, ,
$$

Here, $\widetilde{\kappa}_{ijk}$ are the triple intersection numbers of
the mirror dual Calabi-Yau threefold $\widetilde{X}$.
Here, we defined

$$
    a_{ij} = \dfrac{1}{2}\begin{cases}
                            \widetilde{\kappa}_{iij} & i\geq j\\[0.3em]
                            \widetilde{\kappa}_{ijj} & i<j
                        \end{cases} \, , \quad 
    b_i = \dfrac{1}{24} \int_{\tilde{D}^i}\, c_2(\widetilde{X})\, , \quad  
    \tilde{\xi}=\frac{\zeta(3)\, \chi(\widetilde{X})}{(2\pi)^3}\, .
$$

The instanton part $F_{\mathrm{inst}}$ of the LCS prepotential $F_{\mathrm{LCS}}$
can be expressed in terms of the periods $X^I=(X^0,X^i)$ as

$$
    F_{\mathrm{inst}}(X) = -\frac{(X^0)^2}{(2\pi\mathrm{i})^3}\, \sum_{q\in\mathcal{M}(\widetilde{X})}\, 
    n_q^{0}\, \text{Li}_3\left (\text{e}^{2\pi \text{i} q_i X^i / X^0}\right )\; , \quad 
    \text{Li}_3\left (x\right )=\sum_{m=1}^{\infty}\, \dfrac{x^{m}}{m^{3}}\, .
$$

Here the sum is performed over all effective curve classes $q\in\mathcal{M}(\widetilde{X})$
in the Mori cone $\mathcal{M}(\widetilde{X})$ of the mirror dual manifold $\widetilde{X}$.
Here, the $n_q^{0}$ are the genus-0 Gopakumar-Vafa (GV) invariants which can be computed
systematically using methods described in [hep-th/9308122](https://arxiv.org/pdf/hep-th/9308122.pdf).

The infinite sum appearing in the poly-logarithm $\text{Li}_3$ can be rewritten to arrive at

$$
    \sum_{q\in\mathcal{M}(\widetilde{X})}\, n_q^{0}\, \text{Li}_3\left (\text{e}^{2\pi \text{i} q_i X^i / X^0}\right )
    = \sum_{q\in\mathcal{M}(\widetilde{X})}\, N_q\, \text{e}^{2\pi \text{i} q_i X^i / X^0}
$$

in terms of genus-0 Gromov-Witten (GW) invariants $N_q$. We typically work with the latter to simplify the calculation.


The above functions are implemented in the class `periods_LCS` which is automatically called in `periods` class if we specify `moduli_space_limit="LCS"`.

#### Example for `periods` class


As an example, let us collect a simple model at LCS with $h^{1,2}=2$, `model_ID=1` and `model_type="KS"`:

In [5]:
periods = jaxvacua.periods(h12=2,model_ID=1,model_type="KS",moduli_space_limit="LCS")
periods

Period calculations for h12=2.

This example has mirror triple intersection numbers

In [6]:
periods.mirror_intersection_numbers

Array([[[9, 3],
        [3, 1]],

       [[3, 1],
        [1, 0]]], dtype=int64)

It turns out that this is precisely the model discussed e.g. in https://inspirehep.net/literature/1772253.

Let us perform a small computation here by verifying that the ISD matrix $\mathcal{M}$ representing the Hodge star $\star$ as a matrix for a given integral basis
is indeed symplectic. First, we evaluate the ISD matrix $\mathcal{M}$ for the projective coordinates $X = (1, i, i)$:

In [7]:
X = 1j*jnp.ones(periods.h12+1)
X = X.at[0].set(1.)
M = periods.ISD_matrix(X,jnp.conj(X))
M

Array([[ 3.41497095-0.j,  1.14132464-0.j,  0.02751866-0.j,
        -0.        +0.j,  0.01834577-0.j,  0.70584577-0.j],
       [ 1.14132464-0.j, 11.91524476-0.j,  3.27391442-0.j,
         0.61885731-0.j,  0.34715844-0.j,  0.72215844-0.j],
       [ 0.02751866-0.j,  3.27391442-0.j,  1.98721179-0.j,
         0.25329242-0.j,  0.55553961-0.j, -1.31946039+0.j],
       [-0.        +0.j,  0.61885731-0.j,  0.25329242-0.j,
         0.34850627-0.j, -0.        +0.j, -0.        +0.j],
       [ 0.01834577-0.j,  0.34715844-0.j,  0.55553961-0.j,
         0.        -0.j,  0.37035974-0.j, -0.87964026+0.j],
       [ 0.70584577-0.j,  0.72215844-0.j, -1.31946039+0.j,
         0.        -0.j, -0.87964026+0.j,  3.12035974-0.j]],      dtype=complex128)

As expected, the output is real as it should be. Now, we verify the following three conditions:

$$
\mathcal{M} = \mathcal{M}^T\; ,\quad \mathcal{M}^T\Sigma\mathcal{M} = \Sigma\; ,\quad \mathcal{M}^{-1}=\Sigma^T\mathcal{M}\Sigma\, .
$$

Note that the second and third condition are not independent, but we test both regardless.
These three conditions can be written in `jax.numpy` as:

In [8]:
cond1 = M - M.T
cond2 = jnp.matmul(M.T,jnp.matmul(periods.sigma(),M))- periods.sigma()
cond3 = jnp.linalg.inv(M)-jnp.matmul(periods.sigma().T,jnp.matmul(M,periods.sigma()))

The tests then proceed as follows:

In [9]:
tol = 1e-10
test1 = bool(jnp.max(jnp.abs(cond1)) < tol)
test2 = bool(jnp.max(jnp.abs(cond2)) < tol)
test3 = bool(jnp.max(jnp.abs(cond3)) < tol)
test1, test2, test3

(True, True, True)

We see that all three conditions are indeed satisfied. Below, we will use the ISD matrix to test that the ISD condition for Type IIB GKP-type vacua
is indeed satisfied.

A full list of methods and attributes can be found in the accompanying documentation. 

### `css` sub-module (complex structure sector)

This sub-module contains `css` class containing functions to handle the Kähler geometry of the complex structure moduli space.

A parametrisation of the complex structure moduli space $\mathcal{M}_{\text{cs}}(X)$ is conveniently obtained by choosing half of the periods, say $X^{I}$ from above, as projective coordinates.
Away from the locus $X^0=0$, one then defines local affine coordinates

$$
z^{i}=\dfrac{X^{i}}{X^{0}}\; ,\quad  i=1,\ldots ,h^{1,2}_{-}\, .
$$

Typically, we normalise the 3-form $\Omega$ such that $X^0=1$, but our implementation allows for arbitrary choices for the normalisation.
From the `period` class above, we obtain functions depending only on the complex structure moduli $z^i$ by introducing suitable coordinates.
The period vector in terms of these coordinates is again given in terms of the *prepotential* $F(z)$ through (setting $X^{0}=1$)

$$
\Pi=\left (\begin{array}{c}
 2F-z^{i} \partial_{i}F\\ 
 \partial_{i}F\\ 
    1 \\ 
    z^{i}
\end{array} \right )\, .
$$

From this, we can define e.g. the Kähler potential

$$
K = -\log\biggl (-\mathrm{i}\Pi^{\dagger}\Sigma\Pi\biggl )-\log(-\mathrm{i}(\tau-\bar{\tau})
$$

as well as all of its first and second derivatives $\partial_{z^i}K,\partial_{\tau}K,\partial_{z^i}\partial_{z^j}K,\partial_{\tau}\partial_{\tau}K$ etc.
such as the Kähler metric

$$
K_{i\bar{\jmath}} = \partial_{z^i}\partial_{\overline{z}^j}K\; ,\quad K_{\tau\bar{\tau}} = \partial_{\tau}\partial_{\overline{\tau}}K\, .
$$

The `css` class inherits some functions from the parent class ```jaxvacua.periods```.
E.g., the function ```jaxvacua.periods.ISD_matrix``` (which takes as input the periods and their complex conjugates)
decends to ```jaxvacua.css.ISD_matrix``` which now requires as input $z^{i}$ and $\bar{z}^{i}$.
Other functions inherited in this way are the derivatives of the ISD matrix, the gauge kinetic matrix $\mathcal{N}$ (see above) and its derivatives.


#### Choice of gauge

Let us look at the example from above:

In [16]:
css = jaxvacua.css(h12=2,model_ID = 1,model_type="KS")

Per default, the gauge fixing $X^0 = 1$



In [17]:
z = 1j*jnp.ones(css.h12)
M = css.ISD_matrix(z,jnp.conj(z))
# Alias
#M = css.M(z,jnp.conj(z))
M

Array([[ 3.41497095-0.j,  1.14132464-0.j,  0.02751866-0.j,
        -0.        +0.j,  0.01834577-0.j,  0.70584577-0.j],
       [ 1.14132464-0.j, 11.91524476-0.j,  3.27391442-0.j,
         0.61885731-0.j,  0.34715844-0.j,  0.72215844-0.j],
       [ 0.02751866-0.j,  3.27391442-0.j,  1.98721179-0.j,
         0.25329242-0.j,  0.55553961-0.j, -1.31946039+0.j],
       [-0.        +0.j,  0.61885731-0.j,  0.25329242-0.j,
         0.34850627-0.j, -0.        +0.j, -0.        +0.j],
       [ 0.01834577-0.j,  0.34715844-0.j,  0.55553961-0.j,
         0.        -0.j,  0.37035974-0.j, -0.87964026+0.j],
       [ 0.70584577-0.j,  0.72215844-0.j, -1.31946039+0.j,
         0.        -0.j, -0.87964026+0.j,  3.12035974-0.j]],      dtype=complex128)

This should match

In [18]:
X = 1j*jnp.ones(periods.h12+1)
X = X.at[0].set(1.)
M_periods = periods.ISD_matrix(X,jnp.conj(X))
M-M_periods

Array([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],      dtype=complex128)

We can also try a different gauge

In [20]:
# Define different gauge
X0 = 2+1j

css_X0 = jaxvacua.css(h12=2,model_ID = 1,model_type="KS",gauge_choice=X0)

z = 1j*jnp.ones(css.h12)
M = css_X0.ISD_matrix(z,jnp.conj(z))

X = 1j*jnp.ones(periods.h12+1)
X = X.at[0].set(X0)
M_periods = periods.ISD_matrix(X,jnp.conj(X))
M-M_periods

Array([[ 8.66884242-3.89984376e-15j,  1.15316052+4.92212771e-15j,
         0.38638073+4.46000050e-15j,  0.21868256+2.81712017e-16j,
         1.09074383+2.61250560e-15j,  0.05949383-7.76803615e-15j],
       [ 1.15316052+2.25738106e-15j,  0.36106325+5.82830815e-15j,
        -0.30355301-4.66115857e-16j, -1.25546635+3.38442228e-16j,
         0.15473031-7.85156521e-16j,  0.09223031+2.80243828e-15j],
       [ 0.38638073+2.28261651e-15j, -0.30355301-7.32955641e-15j,
        -0.67337803-3.27477912e-15j, -0.34292185-3.74962638e-16j,
        -0.54675028-1.44561845e-15j,  1.76574972+4.09114288e-15j],
       [ 0.21868256-2.88793754e-17j, -1.25546635-7.96444804e-17j,
        -0.34292185+1.80879762e-17j,  0.24024172-1.66047934e-17j,
        -0.02165291+3.78478689e-17j, -0.02165291-1.18409270e-16j],
       [ 1.09074383+2.14279047e-15j,  0.15473031-5.55708017e-15j,
        -0.54675028-2.70845266e-15j, -0.02165291-2.78279964e-16j,
        -0.35006491-1.27225338e-15j,  1.52493509+3.71947678e-15j],
     

We find agreement again!

#### Computing the Kähler metric

To compute the Käher metric, we use

In [21]:
# Choose point
z = 1j*jnp.ones(css.h12)
tau = 1j

# Evaluate metric
km = css.kahler_metric(z,jnp.conj(z),tau,jnp.conj(tau))
km

Array([[0.20497545+0.j, 0.04900986+0.j, 0.        +0.j],
       [0.04900986+0.j, 0.03036055+0.j, 0.        +0.j],
       [0.        +0.j, 0.        +0.j, 0.25      -0.j]],      dtype=complex128)

The inverse metric is computed as follows:

In [22]:
ikm = css.inverse_kahler_metric(z,jnp.conj(z),tau,jnp.conj(tau))
ikm

Array([[  7.94529026-0.j, -12.82577489+0.j,   0.        -0.j],
       [-12.82577489+0.j,  53.6416336 -0.j,   0.        -0.j],
       [  0.        +0.j,   0.        +0.j,   4.        +0.j]],      dtype=complex128)

We can check that it indeed matches the expected form by computing the inverse of the Kähler metric `km` from above:

In [23]:
ikm-jnp.linalg.inv(km)

Array([[0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j]], dtype=complex128)

### `flux_sector` sub-module

#### Flux compactifications of Type IIB string theory

This sub-module contains `flux_sector` class for computations of the flux scalar potential induced by 3-form flux backgrounds. Flux compactifications are a central framework for connecting string theory to lower-dimensional physics. They involve the compactification of the extra spatial dimensions of string theory on a compact manifold, with additional background fluxes stabilising the moduli fields associated with the geometry. In the context of Type IIB string theory, compactification on Calabi-Yau manifolds with background fluxes provides a mechanism to stabilise moduli, a crucial step towards constructing realistic four-dimensional (4D) effective theories.

Type IIB string theory in ten dimensions contains two types of bosonic fields relevant for flux compactifications: the Neveu-Schwarz–Neveu-Schwarz (NSNS) 2-form field $B_2$ and the Ramond-Ramond (RR) fields $C_0$, $C_2$, and $C_4$. The dynamics of these fields are encoded in the 10D Type IIB supergravity action. Compactifying the theory on a six-dimensional Calabi-Yau threefold $X$ leads to an effective 4D theory.

A Calabi-Yau threefold is a complex three-dimensional Kähler manifold with a vanishing first Chern class. Such manifolds admit a unique holomorphic $(3,0)$-form $\Omega$ and a Kähler form $J$. The metric on the internal space can be written as
\begin{equation}
\mathrm{d}s^2_{10} = \mathrm{e}^{2A(y)} g_{\mu\nu} \mathrm{d}x^\mu \mathrm{d}x^\nu + \mathrm{e}^{-2A(y)} g_{mn} \mathrm{d}y^m \mathrm{d}y^n\, ,
\end{equation}
where $g_{\mu\nu}$ is the 4D metric, $g_{mn}$ is the metric on $X$, and $A(y)$ is the warp factor. Luckily, for what we are about to do, information about the CY metric $ g_{mn} $ and the warp factor $ A $ are mostly irrelevant.

It should be noted that there has been significant progress in constructing CY metrics $ g_{mn} $ numerically using machine learning tools. Many open-source software packages have been developed over time, such as [cymetric](https://github.com/pythoncymetric/cymetric), [cyjax](https://cyjax.readthedocs.io/en/latest/), and [cymyc](https://github.com/Justin-Tan/cymyc). This is another interesting use case for applying neural networks (NNs) to the string landscape that, unfortunately, we will not have time to cover in this lecture.

In flux compactifications, the background fluxes are introduced as integrals of field strengths over the cycles $\Sigma_\alpha$ of the Calabi-Yau manifold. The relevant field strength is the combined 3-form flux
\begin{equation}
G_3 = F_3 - \tau H_3,
\end{equation}
where $F_3 = \mathrm{d}C_2$ is the RR 3-form flux, $H_3 = \mathrm{d}B_2$ is the NSNS 3-form flux, and $\tau = C_0 + \mathrm{i} \, \mathrm{e}^{-\phi}$ is the axio-dilaton.
The associated NSNS- and RR-fluxes $(h_\alpha,f_\alpha)$ must satisfy the quantisation condition
\begin{equation}
h_{\alpha}=\frac{1}{(2\pi)^2 \alpha'} \int_{\Sigma_\alpha} H_3 \in \mathbb{Z}\, , \quad f_{\alpha}=\frac{1}{(2\pi)^2 \alpha'} \int_{\Sigma_\alpha} F_3 \in \mathbb{Z}\, ,
\end{equation}
where $\Sigma_\alpha$ are 3-cycles in $X$.
For general Calabi-Yau compactifications, the number of such fluxes is given by $4(h^{1,2}+1)$ where $h^{1,2}$ is a topological invariant, a so-called **Hodge number**, which counts the number of complex structure deformations of the Calabi-Yau threefold. Each such deformation mode is associated with a separate scalar field, i.e., a modulus $z^i$ which appears in the EFT.


The dynamics of moduli for given fluxes are governed by the [Gukov-Vafa-Witten (GVW) superpotential](https://inspirehep.net/literature/501505)  
\begin{equation}
W_{\text{GVW}}(\tau,Z)=  \int_{X}\,   G_{3}\wedge\Omega_{3} = \left (f-\tau h\right )^{T}\cdot \Sigma\cdot   \Pi(Z)\, .
\end{equation}
The associated Kähler potential is given by  
\begin{equation}
K = -\log\left(-\mathrm{i} \int_X \Omega \wedge \bar{\Omega} \right) - \log\left(-\mathrm{i}(\tau - \bar{\tau})\right).
\end{equation}
The scalar potential is then given by
\begin{equation}
V_{\text{Flux}}= \mathrm{e}^{K} \left( K^{\tau\overline{\tau}}D_{\tau} W\, D_{\overline{\tau}}\overline{ W}+ K^{i\bar\jmath}D_{i} W\, D_{\bar\jmath}\overline{ W} \right)\, ,
\end{equation}
where 
\begin{equation}
D_\tau W = \partial_\tau W + (\partial_\tau K)W \, , \quad D_i W = \partial_i W + (\partial_i K)W \, ,
\end{equation}
are the Kähler covariant derivatives. A vacuum solution is a minimum (or ground state) of the potential $V$ with $\partial_\tau V=\partial_{i}V=0$. Supersymmetric vacua correspond to solutions to the $F$-term equations 
\begin{equation}
D_\tau W = 0 \, ,\quad D_i W = 0 \, .
\end{equation}
These conditions can equivalently written as **ISD condition**, namely
$$
m_{J}-\tau n_{J}-\overline{\mathcal{N}}_{JI}\left (m^{I}-\tau n^{I}\right ) =0\quad\Leftrightarrow\quad f=\left (\mathcal{M}\Sigma s+1c_{0}\right )h
$$
For solutions of this type, the vacuum energy $V_0=\langle V\rangle=0$ vanishes, i.e., we find a flat Minkowski minimum.


#### Example for `flux_sector` class

Let us again look at our example from above:

In [24]:
model = jaxvacua.flux_sector(h12=2,model_ID = 1,model_type="KS",maximum_degree=0)
model

Flux sector with h12=2 complex structure moduli in the LCS limit.

The flux sector is  a subclass of `complex_structure_sector` and thus has access to objects like the Kähler metric etc.
In addition, as the name suggests, it implements functions that depend on the 3-form fluxes like the GVW superpotential, the flux scalar potential and more.
The flux sector still has the period sector as an attribute which can be called via `model.periods` which e.g. in the LCS limit contains also information about the topology of the mirror CY (like triple intersection numbers, GVs, etc.).

Now, to showcase the basic functionality, let us look at a specific flux vacuum obtained in [1912.10047](https://arxiv.org/abs/1912.10047)

In [25]:
fluxes=jnp.array([7, 3, -24, 0, -16, 50,0, 3, -4, 0, 0, 0])
u1sol=2.74215479602462524879172086700112955631003945168828832743217138983767*1j
u2sol=2.05661613496943436323419976712599580262262253939859294519039244649420*1j
tau0 = 6.85540179778358427172610564536555609784128313762349971439377181031816*1j

z0 = jnp.array([u1sol,u2sol])
ctau0 = jnp.conj(tau0)
cz0 = jnp.conj(z0)

We can check that this solution corresponds to an actual vacuum satisfying the $F$-term conditions

In [26]:
model.DW(z0,cz0,tau0,ctau0,fluxes)

Array([0.-4.43472649e-05j, 0.+5.86236724e-05j, 0.+1.51803721e-07j],      dtype=complex128)

We see that the $F$-term conditions are approximately satisfied. This is because above we neglected corrections from worldsheet instanton
corrections. These can be included by increasing the value for `maximum_degree`

In [27]:
model_inst = jaxvacua.flux_sector(h12=2,model_ID = 1,model_type="KS",maximum_degree=2)
model_inst.DW(z0,cz0,tau0,ctau0,fluxes)

Array([0.+5.46806562e-10j, 0.+1.56138475e-09j, 0.+1.17165726e-11j],      dtype=complex128)

We see that the $F$-term conditions including instanton corrections are better approximated by the above solutions.

Other functions pre-implemented include $\partial_i D_j W$

In [28]:
model.dDW(z0,cz0,tau0,ctau0,fluxes)

Array([[ 6.00001935e+00-2.88657986e-15j,  1.99997442e+00+0.00000000e+00j,
        -3.00000007e+00+0.00000000e+00j],
       [ 2.00000619e+00-1.55431223e-15j, -1.60000082e+01-4.00822792e-32j,
         3.99999998e+00+0.00000000e+00j],
       [-2.99999677e+00-2.49800181e-16j,  3.99999572e+00+0.00000000e+00j,
        -1.10718325e-08+0.00000000e+00j]], dtype=complex128)

or $D_i D_j W$

In [29]:
model.DDW(z0,cz0,tau0,ctau0,fluxes)

Array([[ 6.00003870e+00-2.88657986e-15j,  1.99998061e+00+0.00000000e+00j,
        -2.99999683e+00+0.00000000e+00j],
       [ 1.99998061e+00-1.55431223e-15j, -1.60000164e+01-4.00822792e-32j,
         3.99999570e+00+0.00000000e+00j],
       [-2.99999683e+00-2.49800181e-16j,  3.99999570e+00+0.00000000e+00j,
        -2.21436650e-08+0.00000000e+00j]], dtype=complex128)

Moreover, we implemented the standard $F$-term scalar potential induced by 3-form fluxes through the GVW superpotential

In [30]:
model.scalar_potential(z0,cz0,tau0,ctau0,fluxes)

Array(2.84317812e-10-4.28830965e-26j, dtype=complex128)

Moreover, we can compute the Hessian and its eigenvalues

In [31]:
H = model.hessian(z0,cz0,tau0,ctau0,fluxes)
H_evs = jnp.linalg.eigvalsh(H)
H_evs

Array([-4.83899076e-12,  8.17215992e-16,  3.75624779e-01,  3.75625886e-01,
        1.67456255e+01,  1.67456825e+01], dtype=float64)

Two eigenvalues are highly suppressed because at the classical level one complex direction remains flat.

### ```jaxvacua.sampling``` sub-module

We provide a small module to sample values for the moduli and fluxes:

In [32]:
sampler = jaxvacua.data_sampler(model,flux_bounds=[-5,5])

We can then sample initial guesses by calling

In [33]:
moduli,tau,fluxes = sampler.initial_guesses(10)
moduli,tau,fluxes

The sampler is useful when trying to sample flux vacua from ISD sampling.